In [4]:
import pandas as pd

In [5]:
#  read nav_hitory file

nav = pd.read_csv("../data/raw/nav_history.csv")

print(nav.shape)
nav.head()

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
# Convert date

nav["date"] = pd.to_datetime(nav["date"])
nav.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


In [7]:
# sort values

nav = nav.sort_values(
    ["amfi_code", "date"]
)
nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [8]:
# Create daily_return column

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change() * 100
)

nav.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-1.030568
5752,100016,2022-01-05,521.7239,1.286515
5753,100016,2022-01-06,515.7880,-1.137747
5754,100016,2022-01-07,515.1639,-0.120999


In [9]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

print("fact_nav updated successfully")

fact_nav updated successfully


In [11]:
nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

46000

In [10]:
# Remove duplicates:

nav = nav.drop_duplicates()
print("Shape after removing/drop duplicates:", nav.shape)

Shape after removing/drop duplicates: (46000, 4)


In [24]:
# Validate NAV > 0

print((nav["nav"] <= 0).sum())

0


In [25]:
# clean nav_hstroy file saved in processed folder

nav.to_csv("../data/processed/clean_nav.csv", index=False )

print("Saved")

Saved


In [26]:
# Check NAV values

print("NAV <= 0:", (nav["nav"] <= 0).sum())

print("Duplicate Rows:", nav.duplicated().sum())

print("Shape:", nav.shape)

nav.head()

NAV <= 0: 0
Duplicate Rows: 0
Shape: (46000, 3)


,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [15]:
# read investor_transactions file

tx = pd.read_csv(
    "../data/raw/investor_transactions.csv"
)
tx.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [16]:
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [17]:
# Check columns

print(tx.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [25]:

tx["transaction_date"] = pd.to_datetime(
    tx["transaction_date"]
)
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [14]:
# Validate amount

print((tx["amount_inr"] <= 0).sum())

0


In [15]:
# Validate KYC

print( tx["kyc_status"].value_counts())

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [16]:
# clean investor_transactions file saved in processed folder

tx.to_csv( "../data/processed/clean_transactions.csv", index=False)
print("saved")

saved


In [17]:
#  read scheme_performance file

perf = pd.read_csv("../data/raw/scheme_performance.csv")
perf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [20]:
# Negative Sharpe

print(
    (perf["sharpe_ratio"] < 0).sum()
)

0


In [21]:
# Expense Ratio

print(
    perf["expense_ratio_pct"].describe()
)

count    40.000000
mean      1.237000
std       0.386584
min       0.550000
25%       0.787500
50%       1.425000
75%       1.540000
max       1.640000
Name: expense_ratio_pct, dtype: float64


In [27]:
# 

perf.to_csv("../data/processed/clean_performance.csv",
    index=False
)
print("saved")

saved


In [12]:
import pandas as pd

df = pd.read_sql("SELECT * FROM dim_fund LIMIT 5", engine)
print(df.columns)

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='object')
